In [27]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [28]:
import os
import pandas as pd
import openai
from dotenv import load_dotenv

# loading .env file
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

# intialising OpenAI client and checking for API key
client = openai.OpenAI(api_key=api_key)

if api_key:
    print("API key loaded, environment is ready.")
else:
    print("API key not found, check your .env file.")

API key loaded, environment is ready.


In [29]:
def analyze_student_data(file_path):
    df = pd.read_csv(file_path)

    # Analysing the iq scores and test scores only:
    numeric_df = df.select_dtypes(include=["number"])

    stats_results = {}

    for col in numeric_df.columns:
        mean_val = numeric_df[col].mean()
        median_val = numeric_df[col].median()

        # finding outliers using IQR
        Q1 = numeric_df[col].quantile(0.25)
        Q3 = numeric_df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        outliers = numeric_df[
            (numeric_df[col] < lower_bound) | (numeric_df[col] > upper_bound)
        ]

        stats_results[col] = {
            "mean": round(mean_val, 2),
            "median": round(median_val, 2),
            "outlier_count": len(outliers),
            "outlier_values": outliers[col].tolist(),
        }

    return stats_results


print("Analysis is done.")

Analysis is done.


In [30]:
def generate_student_report(stats_dict, output_file="student_analysis_report.txt"):
    clean_stats_text = ""
    for column, metrics in stats_dict.items():
        clean_stats_text += f"\nCOLUMN: {column}\n"
        clean_stats_text += f"  - Mean Score: {metrics['mean']}\n"
        clean_stats_text += f"  - Median Score: {metrics['median']}\n"
        clean_stats_text += f"  - Number of Outliers: {metrics['outlier_count']}\n"

    prompt = f"""
    You are an Academic Data Analyst. Review these metrics for a class:
    {stats_dict}
    
    Write a 2-paragraph executive summary:
    1. Summarize the average IQ and Test Scores for the class as well as their relationship (if any),
    2. Identify the 'anomalies' in this dataset, and
    3. Suggest possible methods to help students whom you assume to be underperforming.
    """

    # Synthesize with LLM
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You are a professional educational consultant.",
            },
            {"role": "user", "content": prompt},
        ],
    )

    narrative = response.choices[0].message.content

    # Saving the report to a text file
    with open(output_file, "w") as f:
        f.write("=== THE AUTOMATED ANALYST: STUDENT PERFORMANCE ===\n")
        f.write("-" * 50 + "\n")
        f.write(
            f"STATISTICAL SUMMARY:{clean_stats_text}\n"
        )  # Using the clean version here
        f.write("-" * 50 + "\n\n")
        f.write("EXECUTIVE SUMMARY:\n")
        f.write(narrative)

    print(f"The final report has been generated to: {output_file}")
    return narrative

In [31]:
class_stats = analyze_student_data("student_scores.csv")

final_narrative = generate_student_report(class_stats)

print("\n--- REPORT PREVIEW ---")
print(final_narrative)

The final report has been generated to: student_analysis_report.txt

--- REPORT PREVIEW ---
The analysis of the class metrics reveals that the average IQ score is 116.0, with a median IQ score of 113.5. These values fall within the average range and suggest a cognitively capable group of students. The Test Score metrics show a mean of 78.0 and a median of 79.0, indicating that while the students' IQ scores suggest potential academic success, their test scores reflect a level of underachievement. The absence of outliers in both the IQ and Test Score datasets indicates a uniform distribution of scores, suggesting that all students are performing consistently, but not necessarily at their full potential.

Despite the lack of outliers, the relatively lower average test scores compared to the IQ scores suggest that some students may be underperforming. To address this issue, targeted educational interventions could be implemented. Possible methods include small-group tutoring focused on are